In [4]:
import numpy as np
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta
from backtesting import Backtest, Strategy
from backtesting.lib import crossover, plot_heatmaps
from FinMind.data import DataLoader
import helper
import warnings
import os
api = DataLoader()

# === 隱藏所有警告與進度條 ===
warnings.filterwarnings("ignore")
os.environ["BACKTESTING_DISABLE_TQDM"] = "1"
# ============================

In [28]:
today = datetime.today() #datetime.strptime('2025-04-30', "%Y-%m-%d")
today = datetime.strptime('2025-04-30', "%Y-%m-%d")
start_date = (today - relativedelta(years=2)).strftime("%Y-%m-%d")
end_date = today.strftime("%Y-%m-%d")
print('start date', start_date)
print('end date', end_date)

ticker = '2317'
df = api.taiwan_stock_daily(
    stock_id=ticker,
    start_date=start_date,
    end_date=end_date
)
df.set_index('date', inplace=True)
df.index = pd.to_datetime(df.index)
df = df.rename(columns={'max': 'High', 'min': 'Low', 'open': 'Open', 'close': 'Close', 'Trading_Volume': 'Volume'})
df

2026-02-02 14:29:43.525 | INFO     | FinMind.data.finmind_api:get_data:171 - download Dataset.TaiwanStockPrice, data_id: 2317


start date 2023-04-30
end date 2025-04-30


,stock_id,Volume,Trading_money,Open,High,Low,Close,spread,Trading_turnover
date,,,,,,,,,
2023-05-02,2317,50296436,5301337184,104.5,106.0,104.0,106.0,1.5,22045
2023-05-03,2317,27017028,2851879362,105.5,106.0,105.0,105.5,-0.5,10815
2023-05-04,2317,12816496,1348173051,105.5,105.5,105.0,105.0,-0.5,8061
2023-05-05,2317,12635718,1326313874,105.0,105.5,104.5,105.0,0.0,6031
2023-05-08,2317,10633215,1118390878,105.5,106.0,105.0,105.0,0.0,6543
...,...,...,...,...,...,...,...,...,...
2025-04-24,2317,46353045,6354097505,139.0,139.0,135.5,136.5,-2.5,30115
2025-04-25,2317,57201568,7980480834,140.5,141.0,138.5,139.0,2.5,32639
2025-04-28,2317,47321711,6732928742,140.0,143.5,140.0,142.5,3.5,32931


In [6]:
# 假設 df 是你的股價資料表，包含 'High', 'Low', 'Close'
def calculate_atr_stop(df, window=14, multiplier=2.0):
    # 1. 計算真實波幅 (True Range)
    high_low = df['High'] - df['Low']
    high_close = np.abs(df['High'] - df['Close'].shift())
    low_close = np.abs(df['Low'] - df['Close'].shift())
    
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    
    # 2. 計算 ATR (通常使用簡單移動平均或指數移動平均)
    df['ATR'] = tr.rolling(window=window).mean()
    
    # 3. 設定止損位 (以多頭為例：收盤價 - N倍 ATR)
    df['Stop_Loss_Level'] = df['Close'] - (df['ATR'] * multiplier)
    
    return df

# 使用範例
# df = calculate_atr_stop(your_stock_dataframe)

In [8]:
df = calculate_atr_stop(df)
df = df.sort_values(by='date', ascending=False)
df

,stock_id,Volume,Trading_money,Open,High,Low,Close,spread,Trading_turnover,ATR,Stop_Loss_Level
date,,,,,,,,,,,
2026-01-30,2317,72547288,15969960312,223.5,223.5,218.5,220.5,-3.5,68562,5.428571,209.642857
2026-01-29,2317,56613117,12688518717,226.5,227.5,222.0,224.0,-1.5,47910,5.357143,213.285714
2026-01-28,2317,80017265,18150589689,227.0,231.5,224.0,225.5,0.0,76165,5.321429,214.857143
2026-01-27,2317,37140264,8370439856,226.0,227.0,224.0,225.5,1.5,34151,5.500000,214.500000
2026-01-26,2317,39852299,8882149216,222.5,225.0,220.0,224.0,2.5,38954,5.750000,212.500000
...,...,...,...,...,...,...,...,...,...,...,...
2024-02-19,2317,25690866,2629333173,101.5,103.0,101.0,103.0,1.5,13906,NaN,NaN
2024-02-16,2317,36916925,3733038587,101.0,101.5,100.5,101.5,0.5,12607,NaN,NaN
2024-02-15,2317,28273526,2866994311,101.5,102.0,101.0,101.0,-0.5,14939,NaN,NaN


In [11]:
def backtest_stops(df, atr_window=14, atr_multiplier=2.0, fixed_pct=0.05):
    # 計算 ATR
    high_low = df['High'] - df['Low']
    high_close = np.abs(df['High'] - df['Close'].shift())
    low_close = np.abs(df['Low'] - df['Close'].shift())
    df['ATR'] = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1).rolling(window=atr_window).mean()

    # 初始化欄位
    df['Fixed_Stop'] = 0.0
    df['ATR_Stop'] = 0.0
    
    # 模擬進場後的止損線 (以最近一波高點為基準)
    # 這裡演示的是 Trailing Stop (移動止損) 邏輯
    current_fixed_max = 0
    current_atr_max = 0
    
    # 這裡可以透過 loop 或 vector 方式計算，以下為邏輯簡述：
    # Fixed Stop Line = Rolling Max Close * (1 - 0.05)
    # ATR Stop Line = Close - (ATR * 2)
    
    df['Fixed_Stop_Line'] = df['Close'].cummax() * (1 - fixed_pct)
    df['ATR_Stop_Line'] = df['Close'] - (df['ATR'] * atr_multiplier)
    
    return df

# 執行分析
# df_result = backtest_stops(your_foxconn_data)

In [31]:
df_result = backtest_stops(df, atr_multiplier=3.0)
df_result = df.sort_values(by='date', ascending=False)
pd.set_option('display.max_rows', 200)
df_result.head(200)

,stock_id,Volume,Trading_money,Open,High,Low,Close,spread,Trading_turnover,ATR,Fixed_Stop,ATR_Stop,Fixed_Stop_Line,ATR_Stop_Line
date,,,,,,,,,,,,,,
2025-04-30,2317,42090535,5991475670,143.0,144.0,141.5,141.5,-1.5,24152,5.392857,0.0,0.0,215.175,125.321429
2025-04-29,2317,40351039,5750350132,141.5,143.5,141.0,143.0,0.5,23400,6.000000,0.0,0.0,215.175,125.000000
2025-04-28,2317,47321711,6732928742,140.0,143.5,140.0,142.5,3.5,32931,6.714286,0.0,0.0,215.175,122.357143
2025-04-25,2317,57201568,7980480834,140.5,141.0,138.5,139.0,2.5,32639,7.357143,0.0,0.0,215.175,116.928571
2025-04-24,2317,46353045,6354097505,139.0,139.0,135.5,136.5,-2.5,30115,8.107143,0.0,0.0,215.175,112.178571
2025-04-23,2317,65973272,9086566741,136.0,139.5,136.0,139.0,7.0,38655,8.035714,0.0,0.0,215.175,114.892857
2025-04-22,2317,44476006,5940736369,134.5,136.0,131.5,132.0,-4.5,55134,8.000000,0.0,0.0,215.175,108.000000
2025-04-21,2317,32102734,4366804312,135.0,137.0,134.5,136.5,1.0,21917,8.285714,0.0,0.0,215.175,111.642857
2025-04-18,2317,26363008,3564680170,135.0,136.5,133.5,135.5,1.0,21715,8.642857,0.0,0.0,215.175,109.571429
